# 01 · Setup & Environment

> **Module 1 · Composer Fundamentals & Model · Lecture 1.1–1.2**

In this notebook you will:

- Build the mental model of orchestration — and of the tool this course uses instead of a local Airflow install
- Prove, in code, that Airflow is **not** required to learn Airflow
- Discover exactly what is installed on your machine and what is not
- Find the sample dataset and verify its headline numbers
- Understand where Composer fits, and what it will cost you

## 1. The premise: no local Airflow install

Airflow is a large, version-sensitive dependency. Installing it locally means
pinning a Python version, resolving a deep dependency tree, and keeping the
install alive. Most learners hit the same wall: the setup takes longer than
the first lesson, and half the failures are environmental rather than
conceptual.

This course inverts the usual approach:

> **You will learn Airflow by reading Airflow's source of truth, and by
> running the same pipeline logic a real DAG would run — through a small
> local runner that models Airflow's semantics.**

That is why there is no `pip install apache-airflow` step, and why there is
no `airflow` binary in this course. What there *is*:

- a **mini Airflow** in `project/local_runner.py` — a task graph with
  dependencies, XCom, branching, trigger rules, retries and a scheduler loop;
- the **pure pipeline logic** in `project/core/` — every business rule, with
  zero Airflow imports;
- the **real DAG files** in `project/dags/` — the artefact you actually deploy
  to Composer.

The split is the important design decision, and it is the one you should copy
into your own repositories:

```text
project/core/pipeline_logic.py   ← the pipeline (pure, testable, no Airflow)
project/dags/03_etl_bigquery.py  ← the orchestration (Airflow import, DAG object)
```

The first file you can unit-test in milliseconds with Airflow absent. The
second file you can read in five seconds and understand the shape of the
workflow. Neither file requires the other to be *understood*.

## 2. Orchestration vs pipeline — the distinction that decides ownership

Two different jobs, conflated more often than any other architectural mistake
in data engineering:

| | A data pipeline | An orchestrator |
|---|---|---|
| **Job** | transform data | decide *when* and *in what order* |
| **Granularity** | one computation over a dataset | many tasks, many datasets |
| **State** | usually stateless | rich: run history, task states, retries |
| **Failure** | exception | exception *plus* retry, skip, alert, backfill |
| **Example** | `transform_orders()` | the `etl_to_gcs` DAG |

The pipeline is the **engine**. The orchestrator is the **control plane**.
Airflow does not transform anything — it runs your transformations, in the
right order, on the right date, and it knows what happened.

### The failure mode orchestration removes

Not "the script crashed". It is **"the pipeline crashed, and half the system
kept going."** One script fails at 02:14; three downstream scripts start
anyway and produce partial results; nobody notices until the morning dashboard
shows a number that is wrong by eleven percent.

### The DAG mental model

A **DAG** is a Directed Acyclic Graph:

- **Nodes** are tasks — a unit of work with a defined outcome.
- **Edges** are dependencies — "this must finish before that starts".
- **Directed** means the edges have an order: A → B means A first.
- **Acyclic** means no loops, so a topological order always exists and the
  graph is always schedulable.

That "acyclic" property is a feature, not a limitation. It is what makes
"run everything that is ready" a well-defined operation.

```text
        ┌──────────────┐
        │ read_orders  │
        └──────┬───────┘
               │
        ┌──────▼───────────┐
        │ transform_orders │  ← validation happens here
        └──────┬───────────┘
               │
        ┌──────▼────────┐
        │   aggregate   │  ← summarise_by_region()
        └──────┬────────┘
               │
        ┌──────▼────────────┐
        │   write_summary  │  ← render_summary_csv() → GCS
        └───────────────────┘
```

## 3. What is actually installed? Let's find out, rather than assume

The honest way to start is to measure. The cell below inspects the running
interpreter and reports what is present and what is not — including the two
things that matter most for this course: **Airflow** and **Jinja2**.

Everything this notebook (and every other notebook in the course) needs is:

- a bare Python 3 install;
- `jinja2` — for the templating lessons;
- optionally `pandas` / `matplotlib` — used sparingly, never required.

If `jinja2` is missing, install it with `pip install jinja2`. That is the
whole dependency list.

In [ ]:
import importlib.util
import platform
import sys

print("=" * 60)
print("PYTHON ENVIRONMENT")
print("=" * 60)
print(f"executable : {sys.executable}")
print(f"version    : {platform.python_version()}")
print(f"platform   : {platform.system()} {platform.machine()}")

# The one optional dependency the course leans on.
HAS_JINJA = importlib.util.find_spec("jinja2") is not None
print(f"jinja2     : {'installed' if HAS_JINJA else 'MISSING -> pip install jinja2'}")

# The dependency the course deliberately does NOT need.
HAS_AIRFLOW = importlib.util.find_spec("airflow") is not None
print(f"airflow    : {'installed' if HAS_AIRFLOW else 'not installed (not required)'}")

# Optional extras.
for name in ("pandas", "matplotlib", "pytest", "nbformat"):
    found = importlib.util.find_spec(name) is not None
    print(f"{name:<11}: {'installed' if found else 'not installed'}")

if HAS_JINJA:
    import jinja2
    print(f"jinja2 version: {jinja2.__version__}")

### The one non-negotiable rule

**No cell in this course may import `airflow`.** Not for a nicer example, not
"just to show the API". Every cell must run on a bare Python 3 install.

We can assert that mechanically rather than trusting ourselves.

In [ ]:
import json
from pathlib import Path

# Find every notebook in the course and confirm none of them import airflow.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "notebooks").exists():
    NOTEBOOK_DIR = Path.cwd().parent

notebook_paths = sorted((NOTEBOOK_DIR / "notebooks").glob("*.ipynb"))
print(f"found {len(notebook_paths)} notebooks in {NOTEBOOK_DIR / 'notebooks'}\n")

offenders = []
for path in notebook_paths:
    nb = json.loads(path.read_text(encoding="utf-8"))
    for index, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        for line in source.splitlines():
            stripped = line.strip()
            if stripped.startswith(("import airflow", "from airflow")):
                offenders.append((path.name, index, stripped))

if offenders:
    print("AIRFLOW IMPORTS FOUND (these would break the course premise):")
    for name, index, line in offenders:
        print(f"  {name} cell {index}: {line}")
else:
    print("PASS: no notebook imports airflow. The course premise holds.")

## 4. Where the course files live

Now let's orient ourselves in the filesystem. The path-resolution preamble
below is used by **every** notebook that touches the project, so it works
whether you launched Jupyter from the course root or from inside `notebooks/`.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)
print("project/ present:", (COURSE_ROOT / "project").exists())
print("core/ present:", (COURSE_ROOT / "project" / "core").exists())

The preamble walks *up* from the current working directory until it
finds the folder that contains `project/`, then puts two directories on
`sys.path`:

- `project/core` — where `pipeline_logic.py`, `templating.py` and friends live;
- `project` — where `local_runner.py` lives.

That is why the next import works from any directory.

In [ ]:
import pipeline_logic
import templating

print("pipeline_logic ->", pipeline_logic.__file__)
print("templating     ->", templating.__file__)
print()
print("pipeline_logic exports:")
for name in sorted(n for n in dir(pipeline_logic) if not n.startswith("_")):
    obj = getattr(pipeline_logic, name)
    if callable(obj):
        print(f"  {name}()")

## 5. The mini-runner: a small Airflow you can read in one sitting

`project/local_runner.py` implements the subset of Airflow's semantics that
changes *observable behaviour* in a pipeline:

- dependency ordering (topological walk, recomputed each pass)
- upstream failure propagation (`upstream_failed`, distinct from `skipped`)
- trigger rules (`all_success`, `none_failed`, `all_done`, `always`)
- branch fan-out and downstream skip
- retries with delay
- XCom push/pull

It does **not** implement executors, the metadata database, webserver, or
providers. Those are exactly the parts you do not need in order to reason
about pipeline *shape* — and they are the parts Composer provides for free.

Let's inspect it rather than take it on faith.

In [ ]:
import inspect

import local_runner

print(f"local_runner.py — {len(inspect.getsource(local_runner).splitlines())} lines\n")

# The classes that model Airflow's core objects.
for name in ("DAG", "Task", "DagRun", "XCom", "TaskSkip", "BranchException"):
    cls = getattr(local_runner, name)
    doc = (cls.__doc__ or "").strip().splitlines()[0]
    print(f"{name:<16} {doc}")

print()
print("Public functions:")
for name in sorted(
    n for n in dir(local_runner)
    if not n.startswith("_") and callable(getattr(local_runner, n))
):
    if name in ("DAG", "Task", "DagRun", "XCom"):
        continue
    print(f"  {name}()")

Let's see the DAGs the runner knows how to execute, and describe the
shape of each one.

In [ ]:
print("Available DAGs:\n")
for name in sorted(local_runner.DAGS):
    print(f"  - {name}")
print()
print("=" * 70)
print(local_runner.DAGS["etl_to_gcs"]().describe())

## 6. The sample dataset

The course ships a synthetic but realistic dataset: **5000 orders** for
`2026-01-15`, across four regions, four currencies, four order statuses and
several channels. It is generated deterministically, so every number in
every chapter and every notebook is reproducible.

Let's look at it.

In [ ]:
import json

SAMPLE_DIR = COURSE_ROOT / "sample_data"
print("sample_data/ contains:")
for path in sorted(SAMPLE_DIR.iterdir()):
    print(f"  {path.name:<26} {path.stat().st_size:>9,} bytes")

print()
print("summary.json:")
print((SAMPLE_DIR / "summary.json").read_text(encoding="utf-8"))

In [ ]:
orders_path = SAMPLE_DIR / "orders.csv"
lines = orders_path.read_text(encoding="utf-8").splitlines()

print(f"orders.csv: {len(lines)} lines (1 header + {len(lines) - 1} data rows)\n")
print("header:")
print("  " + lines[0])
print()
print("first three data rows:")
for line in lines[1:4]:
    print("  " + line)

### Verify the headline numbers

The course states a specific expectation, and we can check it right now:

> 5000 orders → **1245 accepted**, with the region summary
> ```
> region,order_count,revenue_eur
> AMER,318,113592.19
> APAC,285,108194.92
> EMEA,322,118423.22
> LATAM,320,110646.18
> ```

If this cell does not produce exactly that, something is wrong with the
environment and you should stop and fix it before continuing.

In [ ]:
from pipeline_logic import (
    transform_orders,
    summarise_by_region,
    render_summary_csv,
    reject_counts,
)

accepted, rejected = transform_orders(lines)

print(f"raw lines        : {len(lines) - 1}")
print(f"accepted orders  : {len(accepted)}")
print(f"rejected rows    : {len(rejected)}")
print()
print("region summary:")
print(render_summary_csv(summarise_by_region(accepted)))

EXPECTED_ACCEPTED = 1245
EXPECTED_SUMMARY = (
    "region,order_count,revenue_eur\n"
    "AMER,318,113592.19\n"
    "APAC,285,108194.92\n"
    "EMEA,322,118423.22\n"
    "LATAM,320,110646.18\n"
)

actual = render_summary_csv(summarise_by_region(accepted))
print(f"accepted == {EXPECTED_ACCEPTED} : {len(accepted) == EXPECTED_ACCEPTED}")
print(f"summary matches expected      : {actual == EXPECTED_SUMMARY}")
if actual != EXPECTED_SUMMARY:
    print("--- expected ---")
    print(EXPECTED_SUMMARY)
    print("--- actual ---")
    print(actual)

### Why 5000 rows yield 1245 accepted rows

This is not a bug — it is the pipeline working. `transform_orders` applies
**two different filters**, and the distinction matters:

1. **Validity** (`classify_row`) — a row must have an order id, a customer id,
   a known region, a known status, a known currency, a parseable timestamp and
   a non-negative amount. Failures are collected as rejects.
2. **Revenue relevance** — only `status == "complete"` rows count toward
   revenue. Rows that are `active`, `cancelled` or `refunded` are *valid*
   but not revenue-bearing.

Let's check where the rows actually go.

In [ ]:
# Count how the raw rows distribute across status and validity.
from collections import Counter

from pipeline_logic import classify_row, try_parse_order_line, REVENUE_STATUSES

status_counts = Counter()
validity_counts = Counter()

for line in lines[1:]:
    row = try_parse_order_line(line)
    if row is None:
        validity_counts["unparseable"] += 1
        continue
    status_counts[row["status"]] += 1
    validity_counts[classify_row(row)] += 1

print("status distribution (all parseable rows):")
for status, count in sorted(status_counts.items()):
    revenue = (
        "revenue-bearing"
        if status in REVENUE_STATUSES
        else "excluded from revenue"
    )
    print(f"  {status:<12} {count:>5}   {revenue}")

print()
print("validity classification:")
for verdict, count in sorted(validity_counts.items()):
    print(f"  {verdict:<12} {count:>5}")

So: **5000 raw rows → 4705 parseable and valid → 1245 with status
`complete`**. The other 3460 valid rows are accepted-but-not-revenue-bearing
(waiting on `active` orders, or `cancelled`/`refunded`).

That is a genuinely important production property: **an order that is
`active` today becomes `complete` tomorrow.** If your pipeline only ever
processed `complete` rows, you would lose the order permanently. This is
exactly why the course's ETL keeps accepted rows and revenue as separate
concepts — see notebook 06 for the full idempotency discussion.

### The malformed rows file

There is also a small file of deliberately broken rows, used to demonstrate
the dead-letter path.

In [ ]:
malformed = (
    (SAMPLE_DIR / "malformed_orders.csv").read_text(encoding="utf-8").splitlines()
)
print("malformed_orders.csv:\n")
for line in malformed:
    print("  " + line)

print()
accepted_m, rejected_m = transform_orders(malformed)
print(f"accepted: {len(accepted_m)}   rejected: {len(rejected_m)}")
print()
print("reject reasons:")
for reason, count in sorted(reject_counts(rejected_m).items()):
    print(f"  {reason:<26} {count}")

Every reject carries a **stable, low-cardinality reason code**. That
matters operationally: the reason is safe as a metric label (you can alert on
"a new reason code appeared"), whereas the raw payload is not.

## 7. Run a pipeline end-to-end, right now

Everything above was inspection. Now let's run the real thing through the
mini-runner: `read_orders → transform_orders → aggregate → write_summary`,
the same shape as the deployed DAG.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

run = local_runner.run_dag(
    local_runner.DAGS["etl_to_gcs"](),
    logical_date="2026-01-15",
    variables=local_runner.DEFAULT_VARIABLES,
    connections=local_runner.DEFAULT_CONNECTIONS,
    verbose=True,
)

print()
print("Task states:")
for task_id, state in run.state.items():
    print(f"  {task_id:<22} {state}")

In [ ]:
summary = run.pull("aggregate", "summary")
print(render_summary_csv(summary))
print()
print("XCom namespace for 'aggregate':")
for key, value in run.xcom.all_for("aggregate").items():
    print(f"  {key} -> {type(value).__name__}")
print()
print("Target GCS URI (as templated):")
print("  " + str(run.pull("write_summary", "gcs_uri")))
print("Local file written:")
print("  " + str(run.pull("write_summary", "local_path")))

And the output it produced:

## 8. Where Composer fits

The local runner is a **teaching model**. Composer is the product.

| Concern | Local runner | Cloud Composer |
|---|---|---|
| Scheduler | in-process loop | Airflow scheduler, highly available |
| Metadata DB | in-memory dicts | Cloud SQL for PostgreSQL |
| Web UI | none | full Airflow UI |
| Executors | one in-process worker | Celery / Kubernetes / Composer workers |
| XCom | Python dict | metadata DB (or GCS backend) |
| Secrets | plain dicts | Airflow Connections, Secret Manager |
| Credentials | none needed | GCP service account |
| Cost | free | **billed** |

The point of the split is that **you learn the semantics here, and you deploy
the artefact there**. Notebooks 02–08 build the mental model and the code;
notebook 09 provisions Composer; notebook 10 deploys and validates.

### What Composer gives you that you cannot get locally, cheaply

- **Managed, versioned Airflow.** Google maintains the Airflow version and
  the Python runtime; you pick the version and the environment size.
- **Zero-downtime DAG deployment.** New DAG code is deployed while the
  scheduler keeps running.
- **Scale.** Add worker nodes without touching the DAG code.
- **Observability.** The full Airflow UI, plus Cloud Logging, Cloud Monitoring
  and Cloud Trace integration.
- **Managed secrets.** Connections and Secret Manager, not `.env` files.

## 9. A quick self-check

Let's assert the environment is fit for the course, programmatically.

In [ ]:
checks = []

checks.append(("Python 3.9+", sys.version_info >= (3, 9)))
checks.append(("jinja2 installed", HAS_JINJA))
checks.append(("airflow NOT required", True))
checks.append(("project/ importable", "pipeline_logic" in sys.modules))
checks.append(("local_runner importable", "local_runner" in sys.modules))
checks.append(("sample orders.csv present", (SAMPLE_DIR / "orders.csv").exists()))
checks.append(("5000 raw orders", len(lines) - 1 == 5000))
checks.append(("1245 accepted orders", len(accepted) == 1245))
checks.append(("region summary matches", actual == EXPECTED_SUMMARY))
checks.append(("4 regions in summary", len(summarise_by_region(accepted)) == 4))

print()
width = max(len(name) for name, _ in checks)
for name, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {name:<{width}}")

failed = [name for name, ok in checks if not ok]
print()
print("ALL CHECKS PASSED" if not failed else f"FAILED: {', '.join(failed)}")

## Exercises

1. **Environment audit.** Write a function `audit_environment()` that returns a dict of booleans covering: Python version, jinja2 present, pandas present, sample data present, and the local runner importable. Print it.
2. **Harder — reject anatomy.** Load `malformed_orders.csv`, call `transform_orders`, and print each rejected row's `reason`, `detail` and the first 60 characters of `raw`. Then explain, in a markdown cell, why the reason codes are low-cardinality but the raw payload is not.
3. **Harder — the 3755-row gap.** Explain in prose exactly where the 5000 − 1245 = 3755 non-accepted rows went. Verify your explanation numerically using the status and validity counters you computed earlier. Include what would happen to an `active` order if the pipeline only ever selected `complete` rows.

<details>
<summary><b>Solution (click to reveal)</b></summary>

```python
# Exercise 1
def audit_environment():
    import importlib.util
    return {
        "python_3_9_plus": sys.version_info >= (3, 9),
        "jinja2": importlib.util.find_spec("jinja2") is not None,
        "pandas": importlib.util.find_spec("pandas") is not None,
        "sample_data": (COURSE_ROOT / "sample_data" / "orders.csv").exists(),
        "local_runner": "local_runner" in sys.modules,
    }

print(audit_environment())
```

```python
# Exercise 2
for item in rejected_m:
    print(f"{item['reason']:<26} detail={item['detail']!r:<14} raw={item['raw'][:60]!r}")
```

The reason codes (`missing_order_id`, `bad_timestamp`, `unknown_region`,
`unknown_currency`, `wrong_field_count`, ...) form a small closed set, so
they are safe as metric labels and safe to alert on. The raw payload is
arbitrary user data — high cardinality, possibly containing PII, and it ends
up in logs and in the dead-letter table. One is a dimension; the other is
a measure.

```python
# Exercise 3
print(len(lines) - 1 - len(accepted))       # 3755 not accepted
print(sum(status_counts.values()))          # 5000 parseable
print(validity_counts["valid"])             # 4705 valid
```

Of the 5000 rows: 295 are not parseable or valid (rejected), 4705 are valid,
and of those 4705 only the 1245 with `status == "complete"` are
revenue-bearing. The remaining 3460 valid rows are `active`, `cancelled` or
`refunded`.

If the pipeline selected only `complete` rows, an order that arrives as
`active` would be dropped on that day's run and — because the raw file is
replaced daily — would never be picked up again. It would be lost
permanently. This is why the course separates *acceptance* from *revenue*:
a correct pipeline keeps the row and computes revenue from it, so yesterday's
`active` order becomes today's revenue when it completes.

</details>